In [ ]:
import os
import uuid
import chromadb     # Vector Database
import pymupdf
from dotenv import load_dotenv
from langchain_core.documents import Document
from pypdf import PdfReader    # Reads PDF
from langchain_text_splitters import RecursiveCharacterTextSplitter     # Text Split into Chunks
from sentence_transformers import SentenceTransformer   # Store Chunks into multi dimaltional Place as numbers
from langchain_groq import ChatGroq     # LLM

In [ ]:
def read_PDF_files(folder_name):
    all_pdfs = [f for f in os.listdir(folder_name) if f.lower().endswith(".pdf")]
    
    if not all_pdfs:
        print(f"No pdf files found in the desired folder {folder_name}")
        return []
    
    all_docs : list[Document] = []
    pdf_count, page_count = 0, 0

    for pdf in all_pdfs:

        pdf_path = os.path.join(folder_name, pdf)
        doc = pymupdf.open(pdf_path)

        for page_num, page in enumerate(doc):
            text = page.get_text()
            all_docs.append(
                Document(
                    page_content= text,
                    metadata = {
                    'source': pdf,
                    'page': page_num,
                    'text_count': len(text)
                    }
            ))
            page_count += 1
        doc.close()
        pdf_count += 1

    print(f"Total fetched PDF: {pdf_count}")
    print(f"Total number of pages: {page_count}")
    return all_docs

In [ ]:
documents = read_PDF_files('pdf')

In [ ]:
def split_chunks(documents, chunk_size=700, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "।", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    for chunk in chunks:
        chunk.metadata['text_count'] = len(chunk.page_content)
    return chunks

In [ ]:
chunked_docs = split_chunks(documents)

In [ ]:
chunked_docs

In [ ]:
len(chunked_docs)

In [ ]:
chunked_docs

In [ ]:
from sentence_transformers.util import pytorch_cos_sim

In [ ]:
class Embedding_manager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        print(f"Loading model...")
        self.model = SentenceTransformer(self.model_name)
        print("Embedding Dimantion:", self.model.get_embedding_dimension())


    def generate_embedding(self, chunked_docs):

        model = SentenceTransformer(self.model_name)
        embedded = model.encode(chunked_docs, show_progress_bar=True)
        print("Embedding Status", embedded.shape)

        return embedded


In [ ]:
# embedding_model = Embedding_manager()
bangla_embedding = Embedding_manager(model_name="shihab17/bangla-sentence-transformer")

In [ ]:
text = [doc.page_content for doc in chunked_docs]
pdf_embeddings = bangla_embedding.generate_embedding(text)

In [ ]:
class VectorStoreManager:
    def __init__(self, persistent_directory = 'vector_db', collection_name = 'pdf_storege'):
        self.persistent_directory = persistent_directory
        self.collection_name = collection_name
        self.collection = None
        self.client = None
        self._initial_connection()

    def _initial_connection(self):
        os.makedirs(self.persistent_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path = self.persistent_directory)
        
        self.collection = self.client.get_or_create_collection(
            name= self.collection_name,
            metadata= {'decsription': 'Vector Database to store pdf informations for RAG.'}
        )

        print('initialized the vector store with collection:', self.collection_name)
        print('docs in collection:', self.collection.count())

    def add_documents(self, documents, embedding_data):
        if len(documents) != len(embedding_data):
            raise ValueError('Documents and Embedding is not same.')
        
        ids = []
        all_metadata = []
        document_content = []
        embedding_list = []

        for idx, (doc, emb) in enumerate(zip(documents, embedding_data)):
            doc_id = f'doc_{uuid.uuid4()}'
            ids.append(doc_id)

            document_content.append(doc.page_content)

            metadata = dict(doc.metadata)
            metadata['document_index'] = idx
            metadata['content_length'] = len(doc.page_content)
            all_metadata.append(metadata)

            embedding_list.append(emb.tolist())

        self.collection.add(
            ids = ids,
            documents=document_content,
            metadatas=all_metadata,
            embeddings=embedding_list
        )
        print('docs in collection:', self.collection.count())
        

In [ ]:
vectore_store = VectorStoreManager()

In [ ]:
vectore_store.add_documents(documents=chunked_docs, embedding_data=pdf_embeddings)

In [ ]:
load_dotenv()

In [ ]:
class RAGRetriver:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def invoke(self, query, top_k = 5, score_threshold = 0.4):
        query_embedding = self.embedding_manager.generate_embedding(chunked_docs = query)

        result = self.vector_store.collection.query(
            query_embeddings = query_embedding,
            n_results = top_k
        )

        all_documents = []
        all_distances = []

        if result:
            for i, val in enumerate(result['distances'][0]):
                if val > score_threshold:
                    all_documents.append(result['documents'][0][i])
                    all_distances.append(val)

        context = "\n\n".join(all_documents) if all_documents else "No relevant context found."
        system_msg = "You are a helpful agent who explains things simply and politely in Bangla languadge"
        human_msg = f"Context:\n{context}\n\nQuestion: {query}"

        llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0.7,
            max_tokens=1024
        )
        response = llm.invoke([
            ("system", system_msg),
            ("human", human_msg)
        ])

        return response.content

In [ ]:
rag = RAGRetriver(bangla_embedding, vectore_store)
output = rag.invoke('অনুপম কে?')

In [ ]:
print(output)